## Wikidata linked data

# Wikidata Linked Data Enrichment & Mapping

This notebook enriches the project's location data by fetching detailed information from Wikidata and generating a visualization map.

**Workflow:**
1.  **Collect IDs**: Scans CSV files in `inferences/wikidata-references/` to gather all unique Wikidata IDs linked in previous analysis steps.
2.  **Fetch Data**: Connects to the **Wikidata API** (using `pywikibot`) to retrieve metadata for each location, including:
    *   **Labels & Descriptions**: In English, Portuguese, and Chinese.
    *   **Coordinates**: Latitude and Longitude.
    *   **Hierarchy**: Administrative territories and Countries.
3.  **Cache Management**: checks a local cache (`locations_wikidata_info.xlsx`) to avoid redundant API calls, updating it only with new entries.
4.  **Visualize**: Generates an interactive map (`map_all_locations.html`) using Plotly to display all geolocated points.

In [1]:
import glob
import pandas as pd

# get list of all csv files in directory `../inferences/wikidata-references/`
csv_files = glob.glob("../inferences/wikidata-references/*.csv")
wikidata_entities = set()
for file in csv_files:
    print(file)
    df = pd.read_csv(file)
    # Assuming the wikidata id column is named 'wikidata_id'
    if 'wikidata_id' in df.columns:
        wikidata_entities.update(df['wikidata_id'].dropna().astype(str))
    print(len(df))
wikidata_entities = list(set(wikidata_entities))
print(len(wikidata_entities))


../inferences/wikidata-references/residences-1701.csv
144
../inferences/wikidata-references/residences-1644-1701.csv
311
../inferences/wikidata-references/locations_names_wikidata.csv
923
../inferences/wikidata-references/residences-1644.csv
215
1078


## Fetch wikidata data

In [2]:
%pip install pywikibot

Note: you may need to restart the kernel to use updated packages.


Get all the wikidata ids in the database


In [3]:
from dehergne_util import locations_wikidata_info_file

# preload dataframe entities_df from xls file "../inferences/locations_wikidata_info.xlsx"
entities_cache = pd.read_excel(locations_wikidata_info_file, index_col=0)


In [4]:
entities_cache.loc['Q1207099',]


chinese_label                                                    临清市
english_label                                                Linqing
portuguese_label                                             Linqing
english_description                                county-level city
chinese_description                                          中国山东省城市
portuguese_description                                           NaN
coordinates                       (36.849444444444, 115.70611111111)
latitude                                                   36.849444
longitude                                                 115.706111
administrative_entity_id                                     Q373564
administrative_entity_label_en                             Liaocheng
administrative_entity_label_zh                                   聊城市
country_id                                                      Q148
country_label                             People's Republic of China
label                             

In [5]:
entities_cache.sample(10)

,chinese_label,english_label,portuguese_label,english_description,chinese_description,portuguese_description,coordinates,latitude,longitude,administrative_entity_id,administrative_entity_label_en,administrative_entity_label_zh,country_id,country_label,label,administrative_entity_type
wikidata_id,,,,,,,,,,,,,,,,
Q17054,馬提尼克,Martinique,Martinica,"island in the Lesser Antilles, overseas region...",位于中美洲加勒比海的法国海外省,"ilha nas Pequenas Antilhas, região ultramarina...","(14.65, -61.015)",14.650000,-61.015000,Q142,France,法國,Q142,France,Martinique / 馬提尼克,NaN
Q209834,帕米耶,Pamiers,Pamiers,"commune in Ariège, France",法国阿列日省市镇与副省会,comuna francesa,"(43.116388888889, 1.6108333333333)",43.116389,1.610833,Q1726630,canton of Pamiers-Est,NaN,Q142,France,Pamiers / 帕米耶,NaN
Q6259,帕維亞,Pavia,Pavia,Italian comune,意大利市镇,comuna italiana,"(45.185280555556, 9.155)",45.185281,9.155000,Q16231,Province of Pavia,帕維亞省,Q153529,Duchy of Milan,Pavia / 帕維亞,NaN
Q41562,杜蒂戈林,Thoothukudi,Thoothukudi,"city in Tamil Nadu, India",NaN,Cidade indiana,"(8.7833333333333, 78.133333333333)",8.783333,78.133333,Q15198,Thoothukudi district,杜蒂戈林縣,Q668,India,Thoothukudi / 杜蒂戈林,NaN
Q28794376,南安镇 (大余县),"Nan'an Town, Dayu",NaN,NaN,NaN,NaN,"(25.3982, 114.36451)",25.398200,114.364510,Q1179258,Dayu County,大余县,Q148,People's Republic of China,"Nan'an Town, Dayu / 南安镇 (大余县)",NaN
Q13619,科森扎,Cosenza,Cosenza,"Italian comune in the province of Cosenza, Cal...",意大利的一个市镇,comuna italiana,"(39.3, 16.25)",39.300000,16.250000,Q16162,Province of Cosenza,科森扎省,Q38,Italy,Cosenza / 科森扎,NaN
Q1362530,介休市,Jiexiu,Jiexiu,"county-level city in Shanxi, China",中国山西省城市,NaN,"(37.02821, 111.91028)",37.028210,111.910280,Q73045,Jinzhong,晋中市,Q148,People's Republic of China,Jiexiu / 介休市,NaN
Q22502,臺灣島,Taiwan Island,Ilha Formosa,island in East Asia; main land of the Republic...,位於西太平洋的東亞島嶼,ilha na Ásia Oriental,"(23.766666666667, 121)",23.766667,121.000000,Q865,Taiwan,中華民國,Q148,People's Republic of China,Taiwan Island / 臺灣島,NaN
Q941967,佩尼謝,Peniche,Peniche,municipality and city in Portugal,葡萄牙城市,município e cidade de Portugal,"(39.362833333333334, -9.388722222222222)",39.362833,-9.388722,Q244512,Leiria,萊里亞區,Q45,Portugal,Peniche / 佩尼謝,NaN


In [6]:
import time
import pywikibot
import pandas as pd
from dehergne_util import locations_wikidata_info_file

site = pywikibot.Site("wikidata", "wikidata")
repo = site.data_repository()
use_cache = True

entities_data = []
entities_from_cache = []
problems = []  # to store any problems encountered during the fetching process
for wikidata_id in wikidata_entities:
    if wikidata_id == 'Q11146687':
        pass
    # check if the wikidata_id is already in the cache
    if use_cache and wikidata_id in entities_cache.index:
        entities_from_cache.append(wikidata_id)
        # print(f"{wikidata_id} already in cache, skipping...")
        continue
    entity_data = {}
    # Fetch the item from Wikidata
    entity_data['wikidata_id'] = wikidata_id
    print(f"Fetching {wikidata_id} from Wikidata...")
    # Use the repository to get the item
    try:
        item = pywikibot.ItemPage(repo, wikidata_id)
    except Exception as e:
        problems.append({'wikidata_id': wikidata_id, 'exception': str(e)})
        continue
    item_dict = item.get()
    entity_data['chinese_label'] = item_dict['labels'].get('zh', None)
    entity_data['english_label'] = item_dict['labels'].get('en', None)
    entity_data['portuguese_label'] = item_dict['labels'].get('pt', None)
    entity_data['english_description'] = item_dict['descriptions'].get('en', None)
    entity_data['chinese_description'] = item_dict['descriptions'].get('zh', None)
    entity_data['portuguese_description'] = item_dict['descriptions'].get('pt', None)
    # get the coordinates if they exist
    coord = None
    if 'P625' in item_dict['claims']:
        coord_claim = item_dict['claims']['P625'][0].getTarget()
        coord = (coord_claim.lat, coord_claim.lon)
    entity_data['coordinates'] = coord
    # store lat and lon in separate keys
    entity_data['latitude'] = coord[0] if coord else None
    entity_data['longitude'] = coord[1] if coord else None
    # get the administrative territorial entity if it exists (P131)
    administrative_entity = None
    administrative_entity_label_en = None
    administrative_entity_label_zh = None
    administrative_entity_id = None
    administrative_entity_claims = item_dict['claims'].get('P131', [])
    if administrative_entity_claims:
        administrative_entity = administrative_entity_claims[0].getTarget()
        administrative_entity_id = administrative_entity.id
        
        # Fetch data once to be efficient
        admin_data = administrative_entity.get()
        
        administrative_entity_label_en = admin_data['labels'].get('en', None)
        administrative_entity_label_zh = admin_data['labels'].get('zh', None)
        type_claims = admin_data['claims'].get('P31', [])
        if type_claims:
            # Get the first 'instance of' value (e.g., "Province")
            type_item = type_claims[0].getTarget()
            try:
                # Fetch the label of the type
                administrative_entity_type = type_item.get()['labels'].get('en', None)
            except Exception:
                pass
    # store administrative information in entity_data
    entity_data['administrative_entity_id'] = administrative_entity_id
    entity_data['administrative_entity_label_en'] = administrative_entity_label_en
    entity_data['administrative_entity_label_zh'] = administrative_entity_label_zh
    entity_data['administrative_entity_type'] = administrative_entity_type # <--- Store it

    # get the country
    country = None
    country_label = None
    country_id = None
    country_claims = item_dict['claims'].get('P17', [])
    if country_claims:
        country = country_claims[0].getTarget()
        country_id = country.id
        country_label = country.get()['labels'].get('en', None)
    entity_data['country_id'] = country_id
    entity_data['country_label'] = country_label
    # append the entity data to the list
    entities_data.append(entity_data)

    print(f"Fetched {wikidata_id} with labels: {entity_data['english_label']}, {entity_data['portuguese_label']}, {entity_data['chinese_label']}, {entity_data['administrative_entity_label_en']}, {entity_data['administrative_entity_label_zh']}, {entity_data['country_label']}")
    time.sleep(0.5)  # to avoid hitting the API too hard

# Convert the list of dictionaries to a DataFrame
entities_df = pd.DataFrame(entities_data)
if len(entities_df) == 0:
    print("No new entities found.")
    entities_df = entities_cache  # use the cached data if no new entities found
else:
    print(f"Fetched {len(entities_df)} new entities from Wikidata.")
    entities_df.set_index('wikidata_id', inplace=True)

# get the rows entities_cache in entities_from_cache list of ids
# this removes from cache entities not in the current run
entities_cache = entities_cache[entities_cache.index.isin(entities_from_cache)]
# append the new data to the existing cache
entities_df = pd.concat([entities_cache, entities_df], axis=0)



Fetching No wikidata from Wikidata...
No new entities found.


In [7]:
# print the problems encountered one per line
for problem in problems:
    print(f"Problem with {problem['wikidata_id']}: {problem['exception']}")

Problem with No wikidata: 'No wikidata' is not a valid item page title


In [8]:
# show duplicates in the entities_df use index
duplicates = entities_df.index[entities_df.index.duplicated()].unique()
duplicates

Index(['Q90', 'Q41185', 'Q869', 'Q12193', 'Q639421', 'Q1352', 'Q622778',
       'Q16572', 'Q15175', 'Q155',
       ...
       'Q1197421', 'Q645183', 'Q128883', 'Q11065314', 'Q1111625', 'Q11643048',
       'Q1197616', 'Q950944', 'Q10945117', 'Q42666'],
      dtype='object', name='wikidata_id', length=1077)

In [9]:
# remove duplicates from entities_df
print(f"Removing {len(duplicates)} duplicates from entities_df, before removal: {len(entities_df)} rows.")
entities_df = entities_df[~entities_df.index.duplicated(keep='first')]
print(f"After removal: {len(entities_df)} rows.")

Removing 1077 duplicates from entities_df, before removal: 2154 rows.
After removal: 1077 rows.


In [10]:
entities_df.info()
entities_df.head(10)

<class 'pandas.core.frame.DataFrame'>
Index: 1077 entries, Q90 to Q42666
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   chinese_label                   1018 non-null   object 
 1   english_label                   1072 non-null   object 
 2   portuguese_label                890 non-null    object 
 3   english_description             1058 non-null   object 
 4   chinese_description             771 non-null    object 
 5   portuguese_description          619 non-null    object 
 6   coordinates                     1038 non-null   object 
 7   latitude                        1038 non-null   float64
 8   longitude                       1038 non-null   float64
 9   administrative_entity_id        973 non-null    object 
 10  administrative_entity_label_en  969 non-null    object 
 11  administrative_entity_label_zh  858 non-null    object 
 12  country_id                      106

,chinese_label,english_label,portuguese_label,english_description,chinese_description,portuguese_description,coordinates,latitude,longitude,administrative_entity_id,administrative_entity_label_en,administrative_entity_label_zh,country_id,country_label,label,administrative_entity_type
wikidata_id,,,,,,,,,,,,,,,,
Q90,巴黎,Paris,Paris,capital city and largest city of France,法國首都及最大都市,capital e maior cidade da França,"(48.85666666666667, 2.352222222222222)",48.856667,2.352222,Q16665915,Grand Paris,大巴黎都会区,Q142,France,Paris / 巴黎,NaN
Q41185,卡昂,Caen,Caen,"commune in Calvados, France",法国卡尔瓦多斯省市镇与省会,Comuna francesa,"(49.18138888888889, -0.3636111111111111)",49.181389,-0.363611,Q700794,arrondissement of Caen,卡昂区,Q142,France,Caen / 卡昂,NaN
Q869,泰國,Thailand,Tailândia,country in Southeast Asia,東南亞國家,país da Ásia,"(14, 101)",14.000000,101.000000,NaN,NaN,NaN,Q869,Thailand,Thailand / 泰國,NaN
Q12193,布雷斯特,Brest,Brest,"port city in the Finistère department, Brittan...",法国菲尼斯泰尔省市镇与副省会,comuna francesa,"(48.39, -4.4869444444444)",48.390000,-4.486944,Q700737,arrondissement of Brest,布雷斯特区,Q142,France,Brest / 布雷斯特,NaN
Q639421,本地治里市,Pondicherry,NaN,"city in the Union Territory of Puducherry, India",NaN,NaN,"(11.93, 79.83)",11.930000,79.830000,Q66743,Puducherry,本地治里,Q668,India,Pondicherry / 本地治里市,NaN
Q1352,金奈,Chennai,Chennai,"city and state capital of Tamil Nadu, India","印度城市,泰米爾納德邦首府","cidade e capital do estado de Tamil Nadu, Índia","(13.0825, 80.275)",13.082500,80.275000,Q15116,Chennai district,金奈縣,Q668,India,Chennai / 金奈,NaN
Q622778,東京,Tonkin,Tonkin,"northern part of Vietnam, to the west of the G...",印度支那的历史地名，位于今日越南北部,NaN,"(21, 106)",21.000000,106.000000,NaN,NaN,NaN,Q881,Vietnam,Tonkin / 東京,NaN
Q16572,广州市,Guangzhou,Cantão,"capital city of Guangdong Province, China",广东省的省会,capital da província chinesa de Cantão,"(23.13, 113.26)",23.130000,113.260000,Q15175,Guangdong,广东省,Q148,People's Republic of China,Guangzhou / 广州市,NaN
Q15175,广东省,Guangdong,Cantão,province of China,中华人民共和国省份,província da China,"(23.4, 113.5)",23.400000,113.500000,Q148,People's Republic of China,中华人民共和国,Q148,People's Republic of China,Guangdong / 广东省,NaN


In [11]:
entities_df.loc['Q1207099',]

chinese_label                                                    临清市
english_label                                                Linqing
portuguese_label                                             Linqing
english_description                                county-level city
chinese_description                                          中国山东省城市
portuguese_description                                           NaN
coordinates                       (36.849444444444, 115.70611111111)
latitude                                                   36.849444
longitude                                                 115.706111
administrative_entity_id                                     Q373564
administrative_entity_label_en                             Liaocheng
administrative_entity_label_zh                                   聊城市
country_id                                                      Q148
country_label                             People's Republic of China
label                             

In [12]:
from dehergne_util import locations_wikidata_info_file
entities_df.to_excel(locations_wikidata_info_file, index=True)
print(f"Saved {len(entities_df)} entities to {locations_wikidata_info_file}")

Saved 1077 entities to ../inferences/wikidata-references/locations_wikidata_info.xlsx


Entries with no coordinates

Ensure we have plotly and nbformat. Restart the kernel if necessary.

In [13]:
%pip install plotly nbformat

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import plotly.express as px
import plotly.io as pio

# Create a 'label' column by concatenating 'english_label' and 'chinese_label'
# Handle potential None values by replacing them with empty strings
entities_df['label'] = entities_df['english_label'].fillna('') + ' / ' + entities_df['chinese_label'].fillna('')

# Drop rows where latitude or longitude is NaN before plotting
entities_df_map = entities_df.dropna(subset=['latitude', 'longitude'])

fig = px.scatter_map(
    entities_df_map,
    lat="latitude",
    lon="longitude",
    hover_name="label",  # Use the new 'label' column for hover text
    text="label",        # Show the label on the map
    zoom=3,
    map_style="open-street-map",

)
fig.update_layout(autosize=True)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
#fig.update_traces(textfont=dict(size=6))
fig.show(config={"responsive": True})

# Export the figure to an HTML file
pio.write_html(fig,
                config={"responsive": True},
                include_plotlyjs=True,
                file='../inferences/map_all_locations.html',
                auto_open=True)